In [ ]:
import numpy as np
import pandas as pd


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# NLP Text Preprocessing

In [ ]:
import re

In [ ]:
import os
print(os.listdir('/content'))

['.config', 'sample_data']


In [ ]:
import zipfile, glob

z = glob.glob('/content/sms+spam+collection (1).zip')[0]
zipfile.ZipFile(z).extractall('/content/data')

In [ ]:
import os
print(os.listdir('/content/data'))

['readme', 'SMSSpamCollection']


In [ ]:
import pandas as pd

df = pd.read_csv(
    '/content/data/SMSSpamCollection',
    sep='\t',
    names=['label', 'message']
)

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
import re

def clean(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['message'] = df['message'].apply(clean)
print(df.head())

  label                                            message
0   ham  go until jurong point crazy available only in ...
1   ham                            ok lar joking wif u oni
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham        u dun say so early hor u c already then say
4   ham  nah i dont think he goes to usf he lives aroun...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

v = TfidfVectorizer()
X = v.fit_transform(df['message'])
y = df['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
pred = model.predict(X_test)
print(pred[:5])

['ham' 'ham' 'ham' 'ham' 'ham']


In [ ]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_test, pred))

0.9766816143497757


In [ ]:
msg = ["URGENT! Claim your free prize now"]
msg = v.transform(msg)
print(model.predict(msg))

['spam']


In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

[[966   0]
 [ 26 123]]


Evaluation Metrics

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       1.00      0.83      0.90       149

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
pred = model.predict(X_test)

In [ ]:
msg = ["Congratulations! You won a free prize"]
msg = v.transform(msg)
print(model.predict(msg))

['ham']


model save

In [ ]:
import joblib

joblib.dump(model, 'spam_model.pkl')
joblib.dump(v, 'tfidf.pkl')

['tfidf.pkl']

Suspicious message explain karna

In [ ]:
msg = "Urgent! Verify your password now!"

print("Warning: Urgent language and password request.")
print("Do not share your password.")

Do not share your password.


In [ ]:
print("Do not click unknown links.")
print("Never share your OTP or password.")
print("Verify the sender before replying.")

Do not click unknown links.
Never share your OTP or password.
Verify the sender before replying.


In [ ]:
response = "I will verify this request through an official source."
print(response)

I will verify this request through an official source.


Q1. Explain why a message may be suspicious.

# Answer: A message may be suspicious if it uses urgent language, asks for personal information  or contains unknown links

Q2. Identify important warning signs
- Urgent words like “Act now”.
- Requests for passwords or OTPs.
- Unknown links.
- Fake prize offers.

Q3. Suggest safe next steps for the user.

## Answer: Do not click unknown links or share personal information. Verify the sender through an official source.

Q4. Generate a safer response when appropriate.

# Answer: “I will verify this request through an official source before taking any action.

In [ ]:
import os

print(os.path.exists('/content/spam_model.pkl'))
print(os.path.exists('/content/tfidf.pkl'))

True
True


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 29.0 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import joblib
import re

model = joblib.load('/content/spam_model.pkl')
vectorizer = joblib.load('/content/tfidf.pkl')

st.title("SMS Spam Detector")
st.write("Check whether a message is Spam or Legitimate.")

message = st.text_area("Paste your message here:")

if st.button("Analyze Message"):
    if message.strip():
        clean_message = message.lower()
        clean_message = re.sub(r'http\S+|www\S+', ' URL ', clean_message)
        clean_message = re.sub(r'[^\w\s]', '', clean_message)

        message_vector = vectorizer.transform([clean_message])
        prediction = model.predict(message_vector)[0]

        if prediction == "spam":
            st.error("Spam Message")
            st.write("Risk Level: High")
            st.write("Safety Advice: Avoid links and never share passwords or OTPs.")
        else:
            st.success

Writing app.py


In [ ]:
!streamlit run app.py &>/content/streamlit.log &
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸your url is: https://famous-turtles-stand.loca.lt
^C


In [ ]:
!streamlit run /content/app.py --server.port 8501



2026-10-19 14:36:56.323 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.195.70:8501

  Stopping...
  Stopping...


In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://thirty-steaks-flow.loca.lt
^C


In [ ]:
!curl -I http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [ ]:
!streamlit run /content/app.py --server.address 0.0.0.0 --server.port 8501



2026-10-19 14:40:47.374 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.195.70:8501

  Stopping...


In [ ]:
import os

print("App:", os.path.exists("/content/app.py"))
print("Model:", os.path.exists("/content/spam_model.pkl"))
print("Vectorizer:", os.path.exists("/content/tfidf.pkl"))

App: True
Model: True
Vectorizer: True


In [ ]:
!pkill -f streamlit

In [ ]:
!cat /content/streamlit.log



2026-10-19 14:27:27.129 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.195.70:8501

  Stopping...
  Stopping...


In [ ]:
!streamlit run /content/app.py --server.address 0.0.0.0 --server.port 8501 > /content/app_log.txt 2>&1 &

In [ ]:
!sleep 3
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Mon, 19 Oct 2026 14:45:33 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6467
last-modified: Mon, 19 Oct 2026 14:25:45 GMT
etag: "048a6099f8d30470b6ca09f6a09b8ef8"
cache-control: no-cache



In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://cool-frogs-fail.loca.lt
^C


In [ ]:
import zipfile
from google.colab import files

with open('/content/requirements.txt', 'w') as f:
    f.write('streamlit\nscikit-learn\njoblib\n')

with zipfile.ZipFile('/content/spam_detector_app.zip', 'w') as z:
    z.write('/content/app.py', 'app.py')
    z.write('/content/spam_model.pkl', 'spam_model.pkl')
    z.write('/content/tfidf.pkl', 'tfidf.pkl')
    z.write('/content/requirements.txt', 'requirements.txt')

files.download('/content/spam_detector_app.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import streamlit as st
import joblib
import re

model = joblib.load("spam_model.pkl")
vectorizer = joblib.load("tfidf.pkl")

st.title("SMS Spam Detector")
st.write("Check whether your message is Spam or Legitimate.")

message = st.text_area("Paste your message here")

if st.button("Analyze Message"):
    if message.strip():

        clean_message = message.lower()
        clean_message = re.sub(
            r"http\S+|www\S+", " URL ", clean_message
        )
        clean_message = re.sub(
            r"[^\w\s]", "", clean_message
        )

        vector = vectorizer.transform([clean_message])
        prediction = model.predict(vector)[0]

        suspicious_words = [
            "urgent", "free prize", "winner",
            "click here", "verify account",
            "password", "otp", "claim now"
        ]

        found = [
            word for word in suspicious_words
            if word in message.lower()
        ]

        if prediction == "spam":
            st.error("Result: SPAM")
            st.write("Risk Level: High")
            st.write("Warning: This message may be unsafe.")
        else:
            st.success("Result: LEGITIMATE")
            st.write("Risk Level: Lower, but stay careful.")

        if found:
            st.subheader("Suspicious Phrases")
            st.write(", ".join(found))
        else:
            st.write("No listed suspicious phrases found.")

        st.subheader("Explanation")
        if prediction == "spam":
            st.write(
                "The model identified this message as spam. "
                "Be careful with unexpected offers and links."
            )
        else:
            st.write(
                "The model identified this message as legitimate. "
                "This does not guarantee the message is safe."
            )

        st.subheader("Safety Recommendations")
        st.write(
            "- Do not open unknown links.\n"
            "- Never share passwords or OTPs.\n"
            "- Verify messages through official channels."
        )

    else:
        st.warning("Please enter a message.")


2026-10-19 15:00:23.853 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 15:00:24.290 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-19 15:00:24.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 15:00:24.291 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 15:00:24.295 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 15:00:24.297 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 15:00:24.299 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-19 15:00:24.300 Thread 'MainThread': mi

In [ ]:
!streamlit run /content/app.py --server.port 8501



2026-10-19 15:01:36.493 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.195.70:8501

  Stopping...
  Stopping...


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import joblib
import re

model = joblib.load('/content/spam_model.pkl')
vectorizer = joblib.load('/content/tfidf.pkl')

def analyze_message(message):
    if not message.strip():
        return "Message likho.", "", "", ""

    clean = message.lower()
    clean = re.sub(r'http\S+|www\S+', ' URL ', clean)
    clean = re.sub(r'[^\w\s]', '', clean)

    vector = vectorizer.transform([clean])
    prediction = model.predict(vector)[0]

    phrases = [
        "urgent", "free prize", "winner", "click here",
        "verify account", "password", "otp", "claim now"
    ]

    found = [p for p in phrases if p in message.lower()]
    suspicious = ", ".join(found) if found else "Koi listed phrase nahi mili."

    if prediction == "spam":
        result = "SPAM"
        risk = "High risk"
        explanation = "Model ne message ko spam classify kiya. Unknown links

SyntaxError: unterminated string literal (detected at line 30) (536309443.py, line 30)

In [ ]:
import gradio as gr
import joblib
import re

model = joblib.load('/content/spam_model.pkl')
vectorizer = joblib.load('/content/tfidf.pkl')

def analyze_message(message):
    if not message.strip():
        return "Message likho.", "", "", ""

    clean = message.lower()
    clean = re.sub(r'http\S+|www\S+', ' URL ', clean)
    clean = re.sub(r'[^\w\s]', '', clean)

    vector = vectorizer.transform([clean])
    prediction = model.predict(vector)[0]

    phrases = [
        "urgent", "free prize", "winner", "click here",
        "verify account", "password", "otp", "claim now"
    ]

    found = [p for p in phrases if p in message.lower()]
    suspicious = ", ".join(found) if found else "Koi listed phrase nahi mili."

    if prediction == "spam":
        result = "SPAM"
        risk = "High risk"
        explanation = (
            "Model ne message ko spam classify kiya. "
            "Unknown links aur offers se ehtiyat karein."
        )
    else

SyntaxError: expected ':' (3506694444.py, line 34)

In [ ]:
import gradio as gr
import joblib
import re

model = joblib.load('/content/spam_model.pkl')
vectorizer = joblib.load('/content/tfidf.pkl')

def analyze_message(message):
    if not message.strip():
        return "Message likho.", "", "", ""

    clean = message.lower()
    clean = re.sub(r'http\S+|www\S+', ' URL ', clean)
    clean = re.sub(r'[^\w\s]', '', clean)

    prediction = model.predict(
        vectorizer.transform([clean])
    )[0]

    phrases = [
        "urgent", "free prize", "winner",
        "click here", "verify account",
        "password", "otp", "claim now"
    ]

    found = [p for p in phrases if p in message.lower()]
    suspicious = ", ".join(found) if found else "No listed phrases found."

    if prediction == "spam":
        result = "SPAM"
        risk = "High risk"
        explanation = "Model ne spam classify kiya. Links aur offers se ehtiyat karein."
    else:
        result = "LEGITIMATE"
        risk = "Lower risk; safety guaranteed nahi."
        explanation = "Model ne legitimate classify kiya. Phir bhi sender verify karein."

    advice = "Unknown links na kholein. Password ya OTP share na karein. Official channel se verify karein."

    return result, risk, suspicious, explanation + "\n" + advice

demo = gr.Interface(
    fn=analyze_message,
    inputs=gr.Textbox(lines=4, label="Paste your SMS"),
    outputs=[
        gr.Textbox(label="Classification"),
        gr.Textbox(label="Risk Level"),
        gr.Textbox(label="Suspicious Phrases"),
        gr.Textbox(label="Explanation and Safety Advice")
    ],
    title="SMS Spam Detector"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a5cbf56cabd5dd2393.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
